# Assignment 2: training pipeline

You're using a notebook for this stage. If you'd rather use a script,
write `train.py` instead and delete this file. `main.py` refuses to run if
it finds both `train.py` and `train.ipynb`, so exactly one of them must exist.

Replace this cell and everything below it with your own code. When run
via `uv run python main.py train`, this notebook must, start to finish, with no
manual steps in between:

- Get your Roboflow API key via `from api_key import load_api_key`, so that it
  works whether the key is in your `.env` file or in the environment, and
  download your dataset version from Roboflow Universe.
- Finetune yolo11n/yolo11s (or the -cls variant) on that dataset, from the
  pretrained checkpoint.
- Evaluate on the test split against the two baselines named in the brief: the
  pretrained COCO model zero-shot, and a trivial predict-nothing / majority
  class baseline.
- Sweep the confidence threshold, compute the expected cost at each point from
  the cost matrix you defined in REPORT.md, and save the sweep.
- Save everything `app.py` needs to build its dashboard without retraining:
  the metrics, the per-class numbers, the cost sweep, and the chosen operating
  point.

This is your real training pipeline. It takes as long as it takes, and it runs
wherever training is practical for you: your own machine, a GPU you have access
to, or a hosted notebook. Nothing re-runs it to check your submission.

The weights it produces are what you commit to `weights/` and what your app
serves. Say in REPORT.md where you trained and what settings you used.

Save the trained model into `weights/`. That is the file you commit and the
file your app loads.

How you structure the code beyond that is your call to make and be able to
defend: file layout, function boundaries, naming.


In [ ]:
import glob
import json
from collections import Counter, defaultdict
from pathlib import Path

import yaml
from roboflow import Roboflow

from api_key import load_api_key

WORKSPACE = "yolov8-z6snb"
PROJECT = "damage-detect-pzgeu"
VERSION = 1

def download():
    rf = Roboflow(api_key=load_api_key())
    ds = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download(
        "yolov11", location="data/damage-detect-v1", overwrite=True
    )
    root = Path(ds.location)
    print("Dataset folder:", root, "->", sorted(p.name for p in root.iterdir()))
    return root


def dataset_stats(root):
    names = yaml.safe_load(open(root / "data.yaml"))["names"]
    stats = {}
    for split in ["train", "valid", "test"]:
        files = glob.glob(str(root / split / "labels" / "*.txt"))
        boxes, areas = Counter(), defaultdict(list)
        damaged = clean_only = empty = 0
        for f in files:
            classes = set()
            for line in open(f):
                parts = line.split()
                if len(parts) < 5:
                    continue
                c = int(parts[0])
                boxes[names[c]] += 1
                areas[names[c]].append(float(parts[3]) * float(parts[4]))
                classes.add(names[c])
            if "Damage-detected" in classes:
                damaged += 1
            elif classes:
                clean_only += 1
            else:
                empty += 1
        stats[split] = {
            "images": len(files),
            "boxes_per_class": dict(boxes),
            "images_with_damage_box": damaged,
            "images_undamaged_only": clean_only,
            "images_with_no_boxes": empty,
            "median_box_area_pct": {
                k: round(sorted(v)[len(v) // 2] * 100, 1) for k, v in areas.items()
            },
        }
    return stats


root = download()
stats = dataset_stats(root)
Path("results").mkdir(exist_ok=True)
json.dump(stats, open("results/dataset_stats.json", "w"), indent=2)
print(json.dumps(stats, indent=2))

In [ ]:
import shutil
import os
from ultralytics import YOLO

MODEL = "yolo11s.pt"
EPOCHS = 30
IMGSZ = 640
BATCH = 16
QUICK_TEST = os.environ.get("QUICK_TEST", "0") == "1"   # real run by default

if QUICK_TEST:
    EPOCHS, FRACTION, IMGSZ, BATCH = 1, 0.05, 320, 4   # tiny run, only to test the code
else:
    FRACTION = 1.0

model = YOLO(MODEL)
model.train(
    data=str(root / "data.yaml"),
    epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH, fraction=FRACTION,
    project=str(Path("runs").resolve()), name="damage", exist_ok=True,
)

Path("weights").mkdir(exist_ok=True)
shutil.copy("runs/damage/weights/best.pt", "weights/best.pt")

In [ ]:
import gc
import numpy as np
import pandas as pd

del model
gc.collect()

test_imgs = sorted(glob.glob(str(root / "test" / "images" / "*")))
if QUICK_TEST:
    test_imgs = test_imgs[:30]   # tiny slice, only to test the code


def has_damage(img_path):
    lab = root / "test" / "labels" / (Path(img_path).stem + ".txt")
    if not lab.exists():
        return 0
    return int(any(line.split()[0] == "0" for line in open(lab) if line.strip()))


y_true = np.array([has_damage(p) for p in test_imgs])


def damage_score(model, img_paths, class_ids):
    scores = []
    for r in model.predict(img_paths, conf=0.001, imgsz=IMGSZ, verbose=False, stream=True):
        s = [float(c) for c, k in zip(r.boxes.conf, r.boxes.cls) if int(k) in class_ids]
        scores.append(max(s) if s else 0.0)
    return np.array(scores)


def at_threshold(y, scores, t=0.5):
    pred = scores >= t
    tp = int((pred & (y == 1)).sum()); fp = int((pred & (y == 0)).sum())
    fn = int((~pred & (y == 1)).sum()); tn = int((~pred & (y == 0)).sum())
    return {"TP": tp, "FP": fp, "FN": fn, "TN": tn,
            "precision": round(tp / (tp + fp), 3) if tp + fp else 0.0,
            "recall": round(tp / (tp + fn), 3) if tp + fn else 0.0,
            "accuracy": round((tp + tn) / len(y), 3)}


# 1) Fine-tuned model
best = YOLO("weights/best.pt")
scores_ft = damage_score(best, test_imgs, {0})
pd.DataFrame({"image": [Path(p).name for p in test_imgs],
              "has_damage": y_true, "score": scores_ft}).to_csv("results/test_scores.csv", index=False)
if QUICK_TEST:
    map50 = map5095 = 0.0   # skipped in the quick test, computed in the real run
else:
    val = best.val(data=str(root / "data.yaml"), split="test", imgsz=IMGSZ,
                   batch=BATCH, verbose=False, plots=False)
    map50, map5095 = float(val.box.map50), float(val.box.map)
del best
gc.collect()

# 2) Zero-shot COCO model: no damage class exists, so it can never flag damage
coco = YOLO("yolo11s.pt")
vehicle_scores = damage_score(coco, test_imgs, {2, 5, 7})   # car, bus, truck
vehicle_share = float((vehicle_scores >= 0.25).mean())
nothing = np.zeros(len(y_true))

# 3) Trivial baseline: always say "not damaged"
results = {
    "finetuned": {**at_threshold(y_true, scores_ft),
                  "mAP50": round(map50, 3), "mAP50-95": round(map5095, 3)},
    "zero_shot_coco": {**at_threshold(y_true, nothing), "mAP50": 0.0, "mAP50-95": 0.0},
    "predict_nothing": {**at_threshold(y_true, nothing), "mAP50": 0.0, "mAP50-95": 0.0},
}
results["zero_shot_coco"]["share_of_test_images_where_it_sees_a_vehicle"] = round(vehicle_share, 3)
json.dump(results, open("results/baselines.json", "w"), indent=2)
print(pd.DataFrame(results).T)

In [ ]:
import json, os, shutil
from pathlib import Path
from ultralytics import YOLO

QUICK = os.environ.get("QUICK_TEST", "0") == "1"
ROOT = Path("data/damage-detect-v1")

if not QUICK:
    best = YOLO("weights/best.pt")
    val = best.val(data=str(ROOT / "data.yaml"), split="test", imgsz=640, batch=16,
                   plots=True, verbose=False, project=str(Path("runs").resolve()),
                   name="test_eval", exist_ok=True)
    names = best.names
    per_class = {}
    for i, c in enumerate(val.box.ap_class_index):
        per_class[names[int(c)]] = {
            "precision": round(float(val.box.p[i]), 3),
            "recall": round(float(val.box.r[i]), 3),
            "AP50": round(float(val.box.ap50[i]), 3),
            "mAP50-95": round(float(val.box.ap[i]), 3),
        }
    out = {
        "positive_class": "Damage-detected",
        "per_class": per_class,
        "confusion_matrix": {
            "labels": [names[0], names[1], "background"],
            "rows_are": "predicted",
            "columns_are": "true",
            "matrix": val.confusion_matrix.matrix.tolist(),
        },
    }
    json.dump(out, open("results/per_class.json", "w"), indent=2)
    for f in ["confusion_matrix.png", "confusion_matrix_normalized.png"]:
        if (Path(val.save_dir) / f).exists():
            shutil.copy(Path(val.save_dir) / f, "results" / Path(f))
    print(json.dumps(out, indent=2))

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- Your assumptions (same as the business section of REPORT.md) ----
FP_COST = 3        # EUR: supervisor reviews a clean car
TP_COST = 3        # EUR: supervisor reviews a damaged car (repair is billed to the customer)
FN_COST = 400      # EUR: damaged car marked clean, repair not billed
TN_COST = 0        # EUR: clean car, not flagged
BASE_RATE = 0.05   # share of returned cars with new damage

if os.environ.get("QUICK_TEST", "0") != "1":
    df = pd.read_csv("results/test_scores.csv")
    y = df["has_damage"].to_numpy()
    s = df["score"].to_numpy()
    n_pos, n_neg = int((y == 1).sum()), int((y == 0).sum())

    def rates(t):
        flag = s >= t
        return (flag & (y == 1)).sum() / n_pos, (flag & (y == 0)).sum() / n_neg

    def cost_per_1000(t, base=BASE_RATE, fn=FN_COST, fp=FP_COST, tp=TP_COST, tn=TN_COST):
        tpr, fpr = rates(t)
        per_car = base * (tpr * tp + (1 - tpr) * fn) + (1 - base) * (fpr * fp + (1 - fpr) * tn)
        return 1000 * per_car

    grid = np.round(np.arange(0.0, 1.0001, 0.01), 2)
    sweep = pd.DataFrame({"threshold": grid})
    sweep["recall"] = [rates(t)[0] for t in grid]
    sweep["false_alarm_rate"] = [rates(t)[1] for t in grid]
    sweep["flagged_per_1000"] = 1000 * (BASE_RATE * sweep["recall"] + (1 - BASE_RATE) * sweep["false_alarm_rate"])
    sweep["cost_per_1000"] = [cost_per_1000(t) for t in grid]
    sweep.round(4).to_csv("results/cost_sweep.csv", index=False)

    lowest = sweep["cost_per_1000"].min()
    best = sweep[np.isclose(sweep["cost_per_1000"], lowest)].iloc[-1]   # ties: the higher threshold
    near = sweep[sweep["cost_per_1000"] <= lowest * 1.05]["threshold"]
    t_best = float(best["threshold"])

    plt.figure(figsize=(8, 4.5))
    plt.plot(sweep["threshold"], sweep["cost_per_1000"], color="#1f4e79")
    plt.scatter([t_best], [lowest], color="red", zorder=3, label=f"chosen threshold {t_best:.2f}")
    plt.axvline(0.5, color="grey", linestyle="--", linewidth=1, label="provisional 0.5")
    plt.xlabel("Confidence threshold")
    plt.ylabel("Expected cost per 1,000 returns (EUR)")
    plt.title(f"Cost vs threshold (base rate {BASE_RATE:.0%}, FN EUR {FN_COST}, FP EUR {FP_COST})")
    plt.legend()
    plt.tight_layout()
    plt.savefig("results/cost_curve.png", dpi=150)
    plt.close()

    # Sensitivity: how does the best threshold move if my assumptions are wrong?
    rows = []
    for base in [0.01, 0.02, 0.05, 0.10, 0.20]:
        for fn in [50, 100, 200, 400, 800]:
            costs = [cost_per_1000(t, base=base, fn=fn) for t in grid]
            rows.append({"base_rate": base, "fn_cost": fn,
                         "best_threshold": float(grid[int(np.flatnonzero(np.isclose(costs, min(costs)))[-1])]),
                         "min_cost_per_1000": round(min(costs), 1)})
    sens = pd.DataFrame(rows)
    sens.to_csv("results/sensitivity.csv", index=False)

    tpr_b, fpr_b = rates(t_best)
    json.dump({
        "assumptions": {"fp_cost": FP_COST, "tp_cost": TP_COST, "fn_cost": FN_COST,
                       "tn_cost": TN_COST, "base_rate": BASE_RATE},
        "chosen_threshold": t_best,
        "cost_per_1000_at_chosen": round(float(lowest), 1),
        "cost_per_1000_at_0.5": round(float(cost_per_1000(0.5)), 1),
        "recall_at_chosen": round(float(tpr_b), 3),
        "false_alarm_rate_at_chosen": round(float(fpr_b), 3),
        "thresholds_within_5pct_of_best": [float(near.min()), float(near.max())],
        "test_images": {"damaged": n_pos, "clean": n_neg},
    }, open("results/operating_point.json", "w"), indent=2)

    print(json.dumps(json.load(open("results/operating_point.json")), indent=2))
    print(sens.pivot(index="base_rate", columns="fn_cost", values="best_threshold"))

In [ ]:
import json, os
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

# ---- Your assumptions ----
FN_COST = 400            # EUR: damaged car passed automatically (missed)
REVIEW_COST = 3          # EUR: a person reviews one car (assumed always correct)
FP_AUTO_COST = 250       # EUR: clean car failed automatically (wrongful charge)
BASE_RATE = 0.05         # share of returned cars with new damage
MAX_REVIEW_SHARE = 0.05  # at most 5% of returns go to a person

if os.environ.get("QUICK_TEST", "0") != "1":
    df = pd.read_csv("results/test_scores.csv")
    y, s = df["has_damage"].to_numpy(), df["score"].to_numpy()
    pos, neg = s[y == 1], s[y == 0]
    grid = np.round(np.arange(0.0, 1.0001, 0.01), 2)

    def zones(scores, lo, hi):
        auto_pass = (scores < lo).mean()
        auto_fail = (scores >= hi).mean()
        return auto_pass, auto_fail, 1 - auto_pass - auto_fail

    rows = []
    for lo in grid:
        for hi in grid[grid >= lo]:
            pp, pf, pr = zones(pos, lo, hi)
            np_, nf, nr = zones(neg, lo, hi)
            cost = 1000 * (BASE_RATE * (pp * FN_COST + pr * REVIEW_COST)
                           + (1 - BASE_RATE) * (nr * REVIEW_COST + nf * FP_AUTO_COST))
            review = BASE_RATE * pr + (1 - BASE_RATE) * nr
            rows.append({"t_low": lo, "t_high": hi, "cost_per_1000": cost, "review_share": review,
                         "missed_per_1000": 1000 * BASE_RATE * pp,
                         "wrong_charges_per_1000": 1000 * (1 - BASE_RATE) * nf})
    sweep = pd.DataFrame(rows)
    sweep.round(4).to_csv("results/two_threshold_sweep.csv", index=False)

    ok = sweep[sweep["review_share"] <= MAX_REVIEW_SHARE + 1e-9]
    chosen = ok.sort_values(["cost_per_1000", "review_share"]).iloc[0]
    free = sweep.sort_values(["cost_per_1000", "review_share"]).iloc[0]
    alone = sweep[np.isclose(sweep["t_low"], sweep["t_high"])].sort_values("cost_per_1000").iloc[0]

    options = {
        "do_nothing": round(1000 * BASE_RATE * FN_COST, 1),
        "human_checks_everything": round(1000 * REVIEW_COST, 1),
        "model_alone": round(float(alone["cost_per_1000"]), 1),
        "model_plus_review": round(float(chosen["cost_per_1000"]), 1),
    }
    frontier = []
    for cap in np.round(np.arange(0.0, 0.2001, 0.01), 2):
        sub = sweep[sweep["review_share"] <= cap + 1e-9]
        frontier.append({"max_review_share": cap, "min_cost_per_1000": round(float(sub["cost_per_1000"].min()), 1)})
    frontier = pd.DataFrame(frontier)
    frontier.to_csv("results/review_tradeoff.csv", index=False)

    plt.figure(figsize=(8, 4.5))
    plt.plot(frontier["max_review_share"] * 100, frontier["min_cost_per_1000"], color="#1f4e79")
    plt.scatter([MAX_REVIEW_SHARE * 100], [chosen["cost_per_1000"]], color="red", zorder=3,
                label=f"chosen: {MAX_REVIEW_SHARE:.0%} reviewed")
    plt.xlabel("Maximum share of returns sent to a person (%)")
    plt.ylabel("Lowest expected cost per 1,000 returns (EUR)")
    plt.title("Cost vs human review (wrong charge EUR 250, missed damage EUR 400)")
    plt.legend(); plt.tight_layout()
    plt.savefig("results/review_tradeoff.png", dpi=150); plt.close()

    out = {
        "assumptions": {"fn_cost": FN_COST, "review_cost": REVIEW_COST, "fp_auto_cost": FP_AUTO_COST,
                        "base_rate": BASE_RATE, "max_review_share": MAX_REVIEW_SHARE,
                        "reviewer_always_correct": True},
        "chosen": {k: round(float(chosen[k]), 4) for k in
                   ["t_low", "t_high", "cost_per_1000", "review_share", "missed_per_1000", "wrong_charges_per_1000"]},
        "without_review_cap": {k: round(float(free[k]), 4) for k in
                               ["t_low", "t_high", "cost_per_1000", "review_share"]},
        "options_cost_per_1000": options,
    }
    json.dump(out, open("results/review_policy.json", "w"), indent=2)
    print(json.dumps(out, indent=2))
    print(frontier[frontier["max_review_share"].isin([0.0, 0.02, 0.05, 0.10, 0.17])])

In [ ]:
import json, os
import pandas as pd

# ---- Your decisions ----
MAX_REVIEW_SHARE = 0.10       # at most 10% of returns go to a person
RETURNS_PER_DAY = 500         # one large branch
MINUTES_PER_REVIEW = 5        # from the business section
PRODUCTIVE_HOURS = 6          # productive hours per person per day (assumption)
BASE_RATE = 0.05
FN_COST = 400

if os.environ.get("QUICK_TEST", "0") != "1":
    sw = pd.read_csv("results/two_threshold_sweep.csv")
    ok = sw[sw["review_share"] <= MAX_REVIEW_SHARE + 1e-9]
    pick = ok.sort_values(["cost_per_1000", "review_share"]).iloc[0]
    alone = sw[sw["t_low"] == sw["t_high"]].sort_values("cost_per_1000").iloc[0]

    reviews_day = RETURNS_PER_DAY * float(pick["review_share"])
    hours_day = reviews_day * MINUTES_PER_REVIEW / 60
    all_hours = RETURNS_PER_DAY * MINUTES_PER_REVIEW / 60

    out = {
        "t_low": float(pick["t_low"]),
        "t_high": float(pick["t_high"]),
        "review_share": round(float(pick["review_share"]), 4),
        "cost_per_1000": round(float(pick["cost_per_1000"]), 1),
        "missed_damages_per_1000": round(float(pick["missed_per_1000"]), 1),
        "wrong_charges_per_1000": round(float(pick["wrong_charges_per_1000"]), 1),
        "returns_per_day": RETURNS_PER_DAY,
        "reviews_per_day": round(reviews_day, 1),
        "review_hours_per_day": round(hours_day, 1),
        "people_needed": round(hours_day / PRODUCTIVE_HOURS, 2),
        "people_if_human_checks_everything": round(all_hours / PRODUCTIVE_HOURS, 1),
        "options_cost_per_1000": {
            "do_nothing": 1000 * BASE_RATE * FN_COST,
            "human_checks_everything": 3000,
            "model_alone": round(float(alone["cost_per_1000"]), 1),
            "model_alone_threshold": float(alone["t_low"]),
            "model_plus_review": round(float(pick["cost_per_1000"]), 1),
        },
        "assumptions": {"minutes_per_review": MINUTES_PER_REVIEW,
                        "productive_hours_per_person": PRODUCTIVE_HOURS,
                        "reviewer_always_correct": True},
    }
    json.dump(out, open("results/operating_policy.json", "w"), indent=2)
    print(json.dumps(out, indent=2))